[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# SQLModel in FastAPI &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the classes, the engine, `get_session`, and the database
with the eight heroes and three teams. Run it first. Every task makes an application of its own, so
that running a cell twice never leaves two copies of a route behind, and the last cell removes the
scratch folder.


In [1]:
import contextlib
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("fastapi", "fastapi==0.141.1"), ("httpx", "httpx==0.28.1")):
    try:
        version(package)
    except PackageNotFoundError:                                    # install what this runtime is missing
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel
from fastapi import Depends, FastAPI, HTTPException
from fastapi.exceptions import ResponseValidationError
from sqlalchemy import event, insert
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import selectinload
from sqlalchemy.pool import StaticPool
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

with warnings.catch_warnings():                                     # starlette warns here on some installations
    warnings.simplefilter("ignore")
    from fastapi.testclient import TestClient

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


class HeroCreate(SQLModel):                                         # what a client may send
    name: str = Field(max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = None
    team_id: int | None = None


class HeroPublic(SQLModel):                                         # what a client may see
    id: int
    name: str
    age: int | None = None


class TeamPublic(SQLModel):
    id: int
    name: str
    headquarters: str


class HeroWithTeam(HeroPublic):                                     # one level deeper, and no further
    team: TeamPublic | None = None


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

def get_session():
    """One session for one request, closed when the request is finished."""
    with Session(engine) as session:
        yield session

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

print("sqlmodel", sqlmodel.__version__, "| fastapi", version("fastapi"), "| httpx", version("httpx"))


sqlmodel 0.0.42 | fastapi 0.141.1 | httpx 0.28.1


**1.** The teams, as a client may see them.


In [2]:
teams_api = FastAPI()


@teams_api.get("/teams", response_model=list[TeamPublic])
def read_teams(session: Session = Depends(get_session)):
    return session.exec(select(Team).order_by(Team.name)).all()


print(TestClient(teams_api).get("/teams").json())


[{'id': 1, 'name': 'Preventers', 'headquarters': 'Sharp Tower'}, {'id': 3, 'name': 'Wakaland Guard', 'headquarters': 'Grand Palace'}, {'id': 2, 'name': 'Z-Force', 'headquarters': "Sister Margaret's Bar"}]


`TeamPublic` has the three fields a client may see, and the heroes are not among them.


**2.** A team created, and a body that is missing something.


In [3]:
class TeamCreate(SQLModel):
    name: str = Field(max_length=50)
    headquarters: str = Field(max_length=60)


create_api = FastAPI()


@create_api.post("/teams", response_model=TeamPublic, status_code=201)
def create_team(arriving: TeamCreate, session: Session = Depends(get_session)):
    team = Team.model_validate(arriving)
    session.add(team)
    session.commit()
    session.refresh(team)
    return team


client = TestClient(create_api)
print("created:", client.post("/teams", json={"name": "Sky Patrol", "headquarters": "Cloud Base"}).json())
refused = client.post("/teams", json={"name": "Night Watch"})
print("refused:", refused.status_code, refused.json()["detail"][0]["loc"], refused.json()["detail"][0]["msg"])


created: {'id': 4, 'name': 'Sky Patrol', 'headquarters': 'Cloud Base'}
refused: 422 ['body', 'headquarters'] Field required


The 422 names the field that was missing, and no database was asked anything.


**3.** One team, and a team that is not there.


In [4]:
one_api = FastAPI()


@one_api.get("/teams/{team_id}", response_model=TeamPublic)
def read_team(team_id: int, session: Session = Depends(get_session)):
    team = session.get(Team, team_id)
    if team is None:
        raise HTTPException(status_code=404, detail="no team with that id")
    return team


client = TestClient(one_api)
print("found    :", client.get("/teams/1").status_code, client.get("/teams/1").json())
print("not found:", client.get("/teams/99").status_code, client.get("/teams/99").json())


found    : 200 {'id': 1, 'name': 'Preventers', 'headquarters': 'Sharp Tower'}
not found: 404 {'detail': 'no team with that id'}


`session.get` answers `None`, and the route is what turns that into a 404 with a message.


**4.** A team with its heroes in it.


In [5]:
class TeamWithHeroes(TeamPublic):
    heroes: list[HeroPublic] = []


nested_api = FastAPI()


@nested_api.get("/teams/{team_id}/heroes", response_model=TeamWithHeroes)
def read_team_with_heroes(team_id: int, session: Session = Depends(get_session)):
    team = session.exec(select(Team).where(Team.id == team_id)
                        .options(selectinload(Team.heroes))).one_or_none()
    if team is None:
        raise HTTPException(status_code=404, detail="no team with that id")
    return team


answer = TestClient(nested_api).get("/teams/1/heroes").json()
print(answer["name"], "has", len(answer["heroes"]), "heroes, the first is", answer["heroes"][0])


Preventers has 4 heroes, the first is {'id': 2, 'name': 'Spider-Boy', 'age': 16}


`HeroPublic` carries no team, so the nesting stops after one level and there is no loop to detect.
`selectinload` put the heroes in memory before the answer was built.


**5.** A hero deleted.


In [6]:
delete_api = FastAPI()


@delete_api.delete("/heroes/{hero_id}", status_code=204)
def delete_hero(hero_id: int, session: Session = Depends(get_session)):
    hero = session.get(Hero, hero_id)
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    session.delete(hero)
    session.commit()


client = TestClient(delete_api)
gone = client.delete("/heroes/8")
print("deleted:", gone.status_code, "| body:", repr(gone.text))
print("again  :", client.delete("/heroes/8").status_code)


deleted: 204 | body: ''
again  : 404


204 means done and nothing to say, so the body is empty; the second attempt finds nothing and
answers 404.


**6.** A client on a database of its own.


In [7]:
def fresh_client(app):
    """A TestClient for an app, on an empty database this process and the app's thread share."""
    test_engine = create_engine("sqlite://", poolclass=StaticPool,
                                connect_args={"check_same_thread": False})
    SQLModel.metadata.create_all(test_engine)

    def override():
        with Session(test_engine) as session:
            yield session

    app.dependency_overrides[get_session] = override
    return TestClient(app), test_engine


listing_api = FastAPI()


@listing_api.post("/heroes", response_model=HeroPublic, status_code=201)
def create_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    hero = Hero.model_validate(arriving)
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


@listing_api.get("/heroes", response_model=list[HeroPublic])
def read_heroes(session: Session = Depends(get_session)):
    return session.exec(select(Hero).order_by(Hero.name)).all()


client, test_engine = fresh_client(listing_api)
for name, secret in (("Deadpond", "Dive Wilson"), ("Spider-Boy", "Pedro Parqueador")):
    client.post("/heroes", json={"name": name, "secret_name": secret})
print(client.get("/heroes").json())

listing_api.dependency_overrides.clear()
test_engine.dispose()


[{'id': 1, 'name': 'Deadpond', 'age': None}, {'id': 2, 'name': 'Spider-Boy', 'age': None}]


An empty database, two requests that fill it, and one that reads it back, with the scratch database
untouched: the application never knew it was talking to something else.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [SQLModel in FastAPI](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/13-sqlmodel-in-fastapi.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
